# Task 3 part 1: Data preparation

Goal: select a diverse, reliable subset of 50 perturbations and compute their mean log2 fold-change profile ("fingerprint") per condition, relative to control cells in the same condition. 10 of these 50 are held out for testing generalization to perturbations never seen during training/tuning.

In [1]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd

In [2]:
DATA_DIR = "/home/ubuntu/data/frangieh"
adata = sc.read_h5ad(f"{DATA_DIR}/rna_qc_filtered.h5ad")
adata

AnnData object with n_obs × n_vars = 213870 × 23711
    obs: 'library_preparation_protocol', 'perturbation_2', 'MOI', 'sgRNA', 'UMI_count', 'guide_id', 'perturbation', 'tissue_type', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation_type_2', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet'
    var: 'ensembl_id', 'ncounts', 'ncells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'

## Filter out perturbations with too few cells

A fold change averaged over very few cells is noisy. Require at least 100 cells in every condition for a perturbation to be considered reliable enough to model.

In [3]:
min_cells_per_cond = 100

cell_counts = (
    # group data by perturbation and condition (only observed pairs) 
    adata.obs.groupby(["perturbation", "perturbation_2"], observed = True)
    .size()
    # pivot perturbation 2 from an index level into columns 
    .unstack(fill_value=0)
)
# leave out all cells with no perturbation (= control cells)
cell_counts = cell_counts.drop(index="control")

# find the minimum cell count across all conditions for a perturbation
min_cells = cell_counts.min(axis=1)
# keep only perturbations for which all conditions have enough cells and safe the indices (perturbation names)
reliable_perturbations = min_cells[min_cells >= min_cells_per_cond].index.tolist()

# some perturbation targets were filtered out of the gene panel entirely during upstream QC
# without the target gene's own expression later models can't use it as a feature, so require it here too
reliable_perturbations = [p for p in reliable_perturbations if p in adata.var_names]

print(f"{len(reliable_perturbations)} of {len(cell_counts)} perturbations retained")

187 of 248 perturbations retained


## Import Clusters from Task 2 in order to strategically subset the perturbations 

Use the Co-culture Leiden (resolution 0.5) clusters from Task 2 (`perturbation_groups.csv`) as a reference for how similar/different perturbations' effects are. Co-culture is used because it's the most granular clustering (11 clusters). This allows to pick perturbations so that they are diverse in their effects and the trained models are most transferable to new data. 

In [4]:
# load leiden clusters identified in Task 2
leiden_clusters = pd.read_csv(f"{DATA_DIR}/perturbation_groups.csv")
# extract the rows of the dataframe that contain cells from co-culture and clustered by leiden
coculture_leiden = leiden_clusters[(leiden_clusters["condition"] == "Co-culture") & (leiden_clusters["method"] == "leiden")]
# extract the column cluster from the rows and set the indices to the perturbation names
coculture_leiden = coculture_leiden.set_index("perturbation")["cluster"]

# subset the cluster labels to only the perturbations with sufficient cell counts
cluster_labels = coculture_leiden.loc[reliable_perturbations]

# inspect cluster labels distribution
cluster_labels.value_counts().sort_index()

cluster
0      2
1     12
2     17
3      8
4      2
5     19
6     24
7     36
8      7
9     57
10     3
Name: count, dtype: int64

## Stratified selection: Select 50 diverse perturbations and then split in training and test data

Sample proportionally from each cluster so the 50 chosen genes mirror the overall cluster distribution, then repeat the same proportional sampling *within* the 50 to pick the 10 held out purely for testing. This keeps both the modeling set and the test set representative of the full diversity of effects.

In [5]:
# create a random number generator 
rng = np.random.default_rng(42)

# proportionally sample n items across the groups in labels (series of cluster labels)
def stratified_sample(labels, n, rng):
    # overall fraction of perturbations to keep (n = target number)
    frac = n / len(labels)
    # create empty list of chosen perturbations
    chosen = []
    # iterate over each group (cluster) in the list of perturbations linked to clusters (cluster number not used = _)
    for _, group in labels.groupby(labels):
        # compute number of perturbations to take from each cluster (between one and len(group))
        k = min(len(group), max(1, round(len(group) * frac)))
        # extend list of chosen perturbation by randomly drawing k perturbations from the respective group
        chosen.extend(rng.choice(group.index, size=k, replace=False))
    # randomly choose n perturbations from the chosen list and drop any access
    chosen = rng.choice(chosen, size=min(n, len(chosen)), replace=False)
    return list(chosen)

# select 50 perturbations proportional to cluster distribution
selected_50 = stratified_sample(cluster_labels, 50, rng)
# select 10 perturbations from the 50 perturbations as test perturbations
test_10 = stratified_sample(cluster_labels.loc[selected_50], 10, rng)
# set remaining 40 perturbations as training perturbations
train_40 = [p for p in selected_50 if p not in test_10]

print(f"selected: {len(selected_50)}, train: {len(train_40)}, held out: {len(test_10)}")

selected: 50, train: 40, held out: 10


## Subset to the cells we actually need

From here on we only ever use control cells (the fingerprint reference) and cells from the 50 selected perturbations -- the other reliable-but-unselected perturbations are never used again. Subsetting to these cells before normalizing and computing HVGs keeps memory manageable and is exactly the subsetting to 50 perturbations the task asks for.

In [6]:
# keep only control cells and cells belonging to one of the 50 selected perturbations
relevant_mask = adata.obs["perturbation"].isin(selected_50) | (adata.obs["perturbation"] == "control")
adata = adata[relevant_mask.values].copy()

adata.shape

(92532, 23711)

## Normalization and storage of different representations of the counts

Store raw counts as well as the normalized counts in separate layers before performing log transformation including a pseudocount of 1. This data is stored in X and used for the selection of HVGs and for training of the model later. 

In [7]:
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
adata.layers["norm"] = adata.X.copy()

sc.pp.log1p(adata)

## Computation of HVGs and subsetting of dataset to HVGs

Working with all genes would make each FC vector mostly noise and unnecessarily expensive to compute and model. Restrict to the top 2000 HVGs so vectors are a manageable size.

HVGs are selected using only cells outside the 10 held-out test perturbations (`test_10`), then the resulting gene list is applied to the full (already-subsetted) dataset, including the test perturbations. This way the gene panel itself never depends on data from the held-out perturbations.

The 50 selected target genes themselves are also forced into the panel (even if not ranked highly variable), so later modeling notebooks can use each target gene's own expression as a feature without having to reload the raw dataset.

In [8]:
# exclude cells belonging to the 10 held-out test perturbations so HVG selection never sees test data
non_test_mask = ~adata.obs["perturbation"].isin(test_10)
adata_hvg_ref = adata[non_test_mask.values].copy()
sc.pp.highly_variable_genes(adata_hvg_ref, n_top_genes=2000)
hvg_genes = adata_hvg_ref.var_names[adata_hvg_ref.var["highly_variable"]]

# force the 50 target genes into the panel too, even if they weren't ranked highly variable,
# so their own expression is available to later models as a feature
# (a few target genes were already filtered out entirely during upstream QC -- e.g. because a
# successfully knocked-out gene can end up too lowly detected to pass a gene-detection filter --
# so only intersect with genes that actually exist in the panel)
target_genes_present = pd.Index(selected_50).intersection(adata.var_names)
panel_genes = hvg_genes.union(target_genes_present)

# apply the training-derived HVG list (plus the 50 target genes) as a column-subset of the full dataset (all cells, including test perturbations)
adata = adata[:, panel_genes].copy()

# only ~2000 genes now, convert the sparse matrix into a full matrix with zeros
adata.layers["norm"] = np.asarray(adata.layers["norm"].todense())

adata.shape

(92532, 2042)

## Compute pseudobulk log2 fold-change fingerprints

For each (perturbation, condition) pair, compare the mean normalized expression of knockout cells to the mean of control cells in the same condition. Compute log2FC (including pseudocount).

(We also tried averaging each cell's own log2FC instead of taking the log-ratio of the two means, but this turned out to be a bad idea: because `log` is concave, the average of many per-cell logs is pulled down by ordinary single-cell dropout -- cells with zero counts purely from shallow sequencing depth, not biology -- and by an amount that depends mostly on the gene's general expression level rather than on which gene was perturbed. That injected a huge *shared, perturbation-independent* component into every fingerprint, which is exactly what made the baseline model look artificially almost perfect. Taking the log-ratio of the two means avoids this, since no per-cell log values are averaged together.)

In [9]:
conditions = adata.obs["perturbation_2"].unique().tolist()
norm = adata.layers["norm"]

# mean normalized expression of control cells, per condition
control_means = {}
for cond in conditions:
    # boolean series that marks each control cell in each condition
    mask = (adata.obs["perturbation_2"] == cond) & (adata.obs["perturbation"] == "control")
    # average expression of each gene across all control cells
    control_means[cond] = norm[mask.values].mean(axis=0)

# log2FC fingerprint per (perturbation, condition), relative to control cells of that condition
pert_FC = {}
# iterate over the 50 selected perturbations and all conditions
for pert in selected_50:
    for cond in conditions:
        mask = (adata.obs["perturbation_2"] == cond) & (adata.obs["perturbation"] == pert)
        if mask.sum() == 0:
            continue
        # average normalized expression values over all cells with the respective perturbation in the condition
        pert_mean = norm[mask.values].mean(axis=0)
        # compute log2FC of all genes for each perturbation under each condition including a pseudocount
        pert_FC[(pert, cond)] = np.log2((pert_mean + 1) / (control_means[cond] + 1))

# convert key tuple from dictionary into two level index with names perturbation and condition
pert_FC_index = pd.MultiIndex.from_tuples(pert_FC.keys(), names=["perturbation", "condition"])
# create a dataframe by stacking the log2FC values for each gene (columns) above each other, indexed by the condition/perturbation pair (rows)
pert_FC_df = pd.DataFrame(np.vstack(list(pert_FC.values())), index=pert_FC_index, columns=adata.var_names)

pert_FC_df.shape

(150, 2042)

## Save outputs for modeling

In [10]:
pert_FC_selected = pert_FC_df.loc[selected_50, :]

# save dataframe with two level index as pickle as it allows to read back in the two level index
pert_FC_selected.to_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
# save the series of perturbations in test and train as csvs 
pd.Series(train_40, name="perturbation").to_csv(f"{DATA_DIR}/task3_train_40.csv", index=False)
pd.Series(test_10, name="perturbation").to_csv(f"{DATA_DIR}/task3_test_10.csv", index=False)

## Save control-cell expression for later models

Later models need each target gene's own baseline expression in control cells (e.g. to build a PCA basis of "normal" variation and project genes onto it). Save the control cells' normalized expression on the same gene panel, so later notebooks don't need to reload the raw dataset.

In [11]:
# boolean mask marking all control cells (across all conditions)
control_mask = adata.obs["perturbation"] == "control"

# one row per control cell, indexed by its condition, columns are the gene panel
control_expression = pd.DataFrame(
    adata.layers["norm"][control_mask.values],
    index=adata.obs.loc[control_mask, "perturbation_2"].values,
    columns=adata.var_names,
)
control_expression.index.name = "condition"

control_expression.to_pickle(f"{DATA_DIR}/task3_control_expression.pkl")
control_expression.shape

(56343, 2042)

In [12]:
# code to load back in the data 
import pandas as pd

DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f'{DATA_DIR}/task3_pert_FC_selected_50.pkl')
train_40 = pd.read_csv(f'{DATA_DIR}/task3_train_40.csv')['perturbation'].tolist()
test_10 = pd.read_csv(f'{DATA_DIR}/task3_test_10.csv')['perturbation'].tolist()
control_expression = pd.read_pickle(f'{DATA_DIR}/task3_control_expression.pkl')

## Discussion (initial draft -- please rewrite)

**What this notebook does:** Loads the full RNA dataset (213,870 cells x 23,711 genes), then narrows it down to a modeling-ready subset:
1. Filters out perturbations that don't have at least 100 cells in every condition, and additionally requires the perturbed gene itself to still be present in the gene panel (needed later as a feature) -- 187 of 248 perturbations survive this filter.
2. Uses Task 2's Co-culture Leiden clusters (resolution 0.5, 11 clusters) to stratify a selection of 50 perturbations, so the chosen genes are diverse in their effects rather than randomly clumped. From these 50, 10 are held out purely for testing (never touched during training or hyperparameter tuning); the remaining 40 are for training/tuning.
3. Subsets the dataset down to only control cells + cells from the 50 selected perturbations (before normalizing), which keeps memory manageable.
4. Normalizes and selects the top 2000 highly variable genes -- computed only from cells outside the 10 held-out test perturbations, so the gene panel itself never depends on test data (a leakage-safe design) -- plus forces in the 50 target genes themselves so their own baseline expression is usable as a feature by later models. Final panel: 2042 genes.
5. Computes each (perturbation, condition) pair's log2 fold-change "fingerprint": the log-ratio of the perturbed cells' mean expression to the same condition's control cells' mean expression (with a pseudocount of 1).
6. Saves everything later notebooks need: the 150-row (50 perturbations x 3 conditions) fingerprint table, the train/test perturbation lists, and the control cells' own baseline expression on the same gene panel (available for future feature engineering, e.g. a gene's own normal expression level/variance/dropout rate).

**A design decision worth calling out:** we initially tried defining the fingerprint as the *average of each individual cell's own log2FC* instead of the log-ratio of the two means. This seemed like the more "single-cell-native" choice, but it turned out to be a mistake: because log is concave, averaging many per-cell log-ratios is pulled down by ordinary single-cell dropout (cells with zero counts from shallow sequencing, not biology) by an amount that depends mostly on a gene's general expression level -- not on which gene was perturbed. This injected a huge shared, perturbation-independent component into every fingerprint, which is exactly what made the (deliberately simplistic) baseline model in Task3_02 look artificially almost perfect (Pearson ~0.97) when it should have looked much more mediocre. We reverted to the log-ratio-of-means definition, which avoids this.